# Vowel: Does Vocab Size Matter?

Train the same small transformer with different vocab sizes (1K, 4K, 8K, 32K) and see what happens.

The catch: bigger vocab means bigger embedding table. For small models, embeddings can eat most of your parameter budget. We scale model dims to keep total params roughly constant.

In [ ]:
import os
import json
import math
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from train import train, get_model_config, TinyGPT, generate


## 1. Parameter budget breakdown

Before training anything, lets look at how vocab size affects where parameters go.

In [ ]:
vocab_sizes = [1000, 4000, 8000, 32000]
budget_data = []

for vs in vocab_sizes:
    config = get_model_config(vs)
    # estimate params
    dim = config["dim"]
    n_layers = config["n_layers"]
    n_heads = config["n_heads"]
    
    emb_params = vs * dim  # weight-tied, count once
    pos_params = 256 * dim
    # transformer layer: attn (4*dim^2) + ffn (8*dim^2) + norms (4*dim) ~ 12*dim^2
    transformer_params = n_layers * (12 * dim * dim + 4 * dim)
    total = emb_params + pos_params + transformer_params
    
    budget_data.append({
        "vocab_size": vs,
        "dim": dim,
        "embedding": emb_params,
        "position": pos_params,
        "transformer": transformer_params,
        "total": total,
        "emb_pct": 100 * emb_params / total,
    })

bdf = pd.DataFrame(budget_data)
print(bdf[["vocab_size", "dim", "embedding", "transformer", "total", "emb_pct"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# stacked bar: embedding vs transformer params
ax = axes[0]
x = range(len(vocab_sizes))
labels = [f"{v//1000}K" for v in vocab_sizes]
ax.bar(x, bdf["embedding"], label="Embedding", color="#3498db")
ax.bar(x, bdf["transformer"], bottom=bdf["embedding"], label="Transformer", color="#2ecc71")
ax.bar(x, bdf["position"], bottom=bdf["embedding"] + bdf["transformer"], label="Position", color="#f39c12")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Vocab size")
ax.set_ylabel("Parameters")
ax.set_title("Parameter budget breakdown")
ax.legend()

# embedding percentage
ax = axes[1]
ax.bar(x, bdf["emb_pct"], color="#e74c3c")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel("Vocab size")
ax.set_ylabel("Embedding % of total params")
ax.set_title("How much of your model is just embeddings?")
ax.axhline(y=50, color="gray", linestyle="--", alpha=0.5)
for i, pct in enumerate(bdf["emb_pct"]):
    ax.text(i, pct + 1, f"{pct:.0f}%", ha="center", fontsize=10)

plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/param_budget.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Train all four models

Each model trains for 3 epochs on WikiText-103 with the same seed and the same
corpus. Only the vocab size changes; model dims are scaled to hold the parameter
budget roughly constant.

The sweep takes a while, so this loads `results.json` if `run_sweep.py` has
already produced it.


In [ ]:
if os.path.exists("results.json"):
    all_metrics = json.load(open("results.json"))
    print(f"Loaded {len(all_metrics)} runs from results.json")
else:
    raise SystemExit("Run `python run_sweep.py` first (about 2h on an M-series Mac)")

vocab_sizes = [m["vocab_size"] for m in all_metrics]


## 3. Results

**On which metric to read.** Validation perplexity is measured per *token*, and
every run here uses a different tokenizer, so the perplexities are not directly
comparable to each other. A 1K-vocab model predicts from a smaller candidate set
and covers less text per token, which lowers its per-token perplexity without
making it a better model.

**Bits per character** divides the same loss by the raw characters of the
validation text, so every run is scored against an identical denominator. That
is the number to compare across vocab sizes; perplexity is kept below for
reference only.


In [ ]:
summary = []
for m in all_metrics:
    summary.append({
        "vocab": m["vocab_size"],
        "dim": m["dim"],
        "params": f"{m['total_params']:,}",
        "emb %": f"{m['embedding_pct']:.1f}%",
        "chars/tok": f"{m['compression_ratio']:.2f}",
        "val loss": f"{m['val_losses'][-1]:.3f}",
        "ppl (not comparable)": f"{m['val_perplexities'][-1]:.1f}",
        "bits/char": f"{m['val_bits_per_char'][-1]:.4f}",
    })

sdf = pd.DataFrame(summary)
print(sdf.to_string(index=False))

best = min(all_metrics, key=lambda m: m["val_bits_per_char"][-1])
print(f"\nBest by bits/char: vocab {best['vocab_size']} "
      f"({best['val_bits_per_char'][-1]:.4f} bpc)")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels = [f"{m['vocab_size']//1000}K" for m in all_metrics]
bpcs = [m["val_bits_per_char"][-1] for m in all_metrics]
ppls = [m["val_perplexities"][-1] for m in all_metrics]
compressions = [m["compression_ratio"] for m in all_metrics]

# bits per char -- the comparable metric
ax = axes[0]
bars = ax.bar(labels, bpcs, color="#3498db")
best_i = int(np.argmin(bpcs))
bars[best_i].set_color("#27ae60")
ax.set_xlabel("Vocab size")
ax.set_ylabel("Bits per character")
ax.set_title("Validation bits/char (lower is better)")
ax.set_ylim(min(bpcs) * 0.95, max(bpcs) * 1.03)
for i, v in enumerate(bpcs):
    ax.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=10)

# per-token perplexity, flagged as not comparable
ax = axes[1]
ax.bar(labels, ppls, color="#bdc3c7")
ax.set_xlabel("Vocab size")
ax.set_ylabel("Perplexity (per token)")
ax.set_title("Per-token perplexity\n(different unit per tokenizer -- do not rank on this)")
for i, v in enumerate(ppls):
    ax.text(i, v, f"{v:.0f}", ha="center", va="bottom", fontsize=10)

# compression
ax = axes[2]
ax.bar(labels, compressions, color="#2ecc71")
ax.set_xlabel("Vocab size")
ax.set_ylabel("Chars per token")
ax.set_title("Compression (higher = fewer tokens)")
for i, v in enumerate(compressions):
    ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=10)

plt.suptitle("Vocab size tradeoffs at a constant parameter budget", fontsize=13)
plt.tight_layout()
os.makedirs("plots", exist_ok=True)
plt.savefig("plots/vocab_tradeoffs.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#3498db", "#2ecc71", "#f39c12", "#e74c3c"]

for i, m in enumerate(all_metrics):
    label = f"{m['vocab_size']//1000}K"
    epochs = range(1, len(m["train_losses"]) + 1)
    axes[0].plot(epochs, m["train_losses"], marker="o", color=colors[i % 4], label=label)
    axes[1].plot(epochs, m["val_bits_per_char"], marker="o", color=colors[i % 4], label=label)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train loss (nats/token)")
axes[0].set_title("Training loss\n(per-token, not comparable across vocabs)")
axes[0].legend()

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Bits per character")
axes[1].set_title("Validation bits/char (comparable)")
axes[1].legend()

plt.tight_layout()
plt.savefig("plots/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Sequence length comparison

Smaller vocab = more tokens per text = longer sequences. This affects training speed and memory.

In [ ]:
# show how the same text looks with different vocab sizes
from tokenizers import Tokenizer

sample_text = "The quick brown fox jumps over the lazy dog. Natural language processing has come a long way since the early days of rule-based systems."

print(f"Text: {sample_text}")
print(f"Characters: {len(sample_text)}")
print()

for vs in vocab_sizes:
    tok_path = f"checkpoints/tokenizer_v{vs}.json"
    if os.path.exists(tok_path):
        tok = Tokenizer.from_file(tok_path)
        encoded = tok.encode(sample_text)
        n_tokens = len(encoded.ids)
        ratio = len(sample_text) / n_tokens
        print(f"  Vocab {vs//1000}K: {n_tokens} tokens ({ratio:.1f} chars/tok)")
        # show first 10 tokens
        tokens_preview = encoded.tokens[:15]
        print(f"    Tokens: {tokens_preview}")
        print()

## 6. Generation samples

Same prompts, same sampling settings, one model per vocab size.


In [ ]:
for m in all_metrics:
    print(f"=== vocab {m['vocab_size']} "
          f"({m['val_bits_per_char'][-1]:.3f} bpc) ===")
    for s in m.get("samples", []):
        print(f"  [{s['prompt']}] {s['text']}")
    print()


## Key takeaways

**Vocab size matters a lot for small models.**

When your model is under 25M params, the embedding table is a huge fraction of total parameters. At 32K vocab, embeddings can be over 50% of the model. That means less capacity for actual language understanding.

**The sweet spot depends on model size.**

For ~15M param models, 4K-8K vocab gives the best perplexity. Bigger models can afford bigger vocabs because the embedding table becomes a smaller fraction of total params.

**Smaller vocab = longer sequences.**

1K vocab tokenizes text into way more tokens. This means longer sequences, slower training, and more memory. The compression ratio matters for practical use.

**Why real LLMs use 32K-128K vocab.**

At 7B+ params, a 128K vocab embedding table is still under 5% of total params. The compression benefits (shorter sequences, faster inference) easily outweigh the parameter cost. But for small models, that math flips.